In [ ]:
# ============================================================
# Cell 0: 环境自检（本章纯 CPU 即可，无需 GPU）
# ============================================================
# 本章只在维度 ≤ 64 的玩具张量上做矩阵乘法、训几十步小网络，全程 CPU 秒级跑完，
# 读 Qwen3 config 也只下载几 KB 的 json，所以不强制 GPU，只打印环境信息确认可用。
import sys, platform
import torch

print("Python:", sys.version.split()[0])
print("平台:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA 可用:", torch.cuda.is_available(), "（本章用不到，CPU 即可）")

In [ ]:
%%capture
# ============================================================
# Cell 1: 安装依赖
# ============================================================
# %%capture 必须是 cell 第一行，把 pip 安装日志藏起来。
# torch:        张量运算 + nn.Module——Colab 默认已装且够新，故意【不】加 -U：
#               会话中途升级 torch 容易让内核半新半旧、后续 import 报错。
# transformers: 第 8.7 节用 AutoConfig 读 Qwen3-8B 的 config（intermediate_size 等）。
#               Qwen3 系列要求 transformers>=4.51，这里锁定版本下界。
!pip install -q -U "transformers>=4.51"

In [ ]:
# ============================================================
# Cell 2: 从零实现 FFN（对应第 2 节）—— 升维 → 激活 → 降维
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F

class FeedForward(nn.Module):
    """原版 Transformer 的 FFN：两层线性 + 一个激活，position-wise。
    x: [B, L, d_model] -> [B, L, d_model]（形状守恒）。"""
    def __init__(self, d_model, d_ff=None, activation="gelu"):
        super().__init__()
        d_ff = d_ff or 4 * d_model            # 经典取 4 倍宽
        self.W1 = nn.Linear(d_model, d_ff)    # 升维 d_model -> d_ff
        self.W2 = nn.Linear(d_ff, d_model)    # 降维 d_ff -> d_model
        self.act = {"relu": F.relu, "gelu": F.gelu}[activation]

    def forward(self, x):
        return self.W2(self.act(self.W1(x)))  # 逐 token 各过各的，同一套权重

torch.manual_seed(0)
B, L, d_model = 2, 5, 32
ffn = FeedForward(d_model, d_ff=128)
x = torch.randn(B, L, d_model)
y = ffn(x)
print("输入 x :", tuple(x.shape), " 输出 y :", tuple(y.shape), "（形状守恒）")

# 验证 position-wise：把第 0 个 token 单独喂进去，结果应和它在整段里的输出一致
y_token0_alone = ffn(x[:, :1, :])             # 只喂第 0 个位置
print("逐位置一致性 (position-wise):",
      torch.allclose(y[:, :1, :], y_token0_alone, atol=1e-6))

# 参数量：FFN 两矩阵 ~ 2·d_model·d_ff，对 d_ff=4·d_model 即 8·d_model²
n_ffn = sum(p.numel() for p in ffn.parameters() if p.requires_grad)
print(f"FFN 参数量: {n_ffn}（含 bias；主项 2·d_model·d_ff = {2*d_model*128}）")

In [ ]:
# ============================================================
# Cell 3: 残差连接验证——深层堆叠下，残差如何救活底层梯度（对应第 3 节）
# ============================================================
class DeepStack(nn.Module):
    """N 层小 MLP 堆叠。use_residual=True 时每层走 x + F(x)，否则只走 F(x)。"""
    def __init__(self, d, n_layers, use_residual):
        super().__init__()
        self.layers = nn.ModuleList(nn.Linear(d, d) for _ in range(n_layers))
        self.act = nn.Tanh()                  # 用 tanh：增益 <1，最容易演示梯度消失
        self.use_residual = use_residual

    def forward(self, x):
        for layer in self.layers:
            fx = self.act(layer(x))
            x = x + fx if self.use_residual else fx   # 唯一区别就在这一行
        return x

def bottom_grad_norm(use_residual, d=32, n_layers=40, seed=0):
    torch.manual_seed(seed)
    net = DeepStack(d, n_layers, use_residual)
    x = torch.randn(4, d)
    loss = net(x).pow(2).mean()               # 任意标量损失
    loss.backward()
    # 看【第 0 层】（最底层）权重的梯度范数——它要从第 40 层一路传回来
    return net.layers[0].weight.grad.norm().item()

g_res = bottom_grad_norm(use_residual=True)
g_plain = bottom_grad_norm(use_residual=False)
print(f"40 层堆叠，最底层权重的梯度范数：")
print(f"  有残差 (x + F(x)) : {g_res:.3e}")
print(f"  无残差 (   F(x) ) : {g_plain:.3e}")
print(f"  比值 有/无 ≈ {g_res / max(g_plain, 1e-30):.1f} 倍")
print("→ 无残差时底层梯度被 40 层连乘衰减到几乎为 0（学不动）；")
print("  有残差时那条 +1 直通路把梯度健康地送回了底层。")

In [ ]:
# ============================================================
# Cell 4: 手写 LayerNorm 与 RMSNorm，并和官方对齐（对应第 4 节）
# ============================================================
class MyLayerNorm(nn.Module):
    """在最后一维（特征维）上标准化：减均值、除标准差，再 γ·x̂ + β。"""
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(d))
        self.beta = nn.Parameter(torch.zeros(d))
        self.eps = eps

    def forward(self, x):
        mu = x.mean(dim=-1, keepdim=True)                 # 每个 token 自己的均值
        var = x.var(dim=-1, keepdim=True, unbiased=False) # 有偏方差（除以 d，与官方一致）
        x_hat = (x - mu) / torch.sqrt(var + self.eps)
        return self.gamma * x_hat + self.beta

class MyRMSNorm(nn.Module):
    """只除以均方根 RMS(x)，不减均值、无 β（LLaMA / Qwen 用的就是这个）。"""
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(d))
        self.eps = eps

    def forward(self, x):
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return self.gamma * (x / rms)                     # 没有减均值，也没有 β

torch.manual_seed(0)
d = 8
x = torch.randn(3, d) * 5 + 2          # 故意给个非零均值、较大尺度的输入

# 1) 和官方 nn.LayerNorm 对齐
mine_ln = MyLayerNorm(d)
ref_ln = nn.LayerNorm(d)               # γ=1, β=0 初始，与我们一致
print("LayerNorm 与官方最大误差:", (mine_ln(x) - ref_ln(x)).abs().max().item())

# 2) 看两种 norm 对「均值 / 尺度」各做了什么
ln_out = mine_ln(x)
rms_out = MyRMSNorm(d)(x)
print(f"\n输入每行均值: {x.mean(-1).tolist()}")
print(f"LayerNorm 后每行均值: {[round(v, 3) for v in ln_out.mean(-1).tolist()]}  ← 被拉到 ~0")
print(f"RMSNorm  后每行均值: {[round(v, 3) for v in rms_out.mean(-1).tolist()]}  ← 没动（不减均值）")
print(f"RMSNorm  后每行 RMS : {[round(v, 3) for v in rms_out.pow(2).mean(-1).sqrt().tolist()]}  ← 被拉到 ~1")

In [ ]:
# ============================================================
# Cell 5: 从零实现 SwiGLU FFN，并和老式 FFN 比参数量（对应第 5 节）
# ============================================================
class SwiGLU(nn.Module):
    """SwiGLU 前馈：( SiLU(x·W_gate) ⊙ (x·W_up) ) · W_down，三个矩阵。"""
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.W_gate = nn.Linear(d_model, d_ff, bias=False)  # 门路径，过 SiLU
        self.W_up = nn.Linear(d_model, d_ff, bias=False)    # 内容路径（升维），不过激活
        self.W_down = nn.Linear(d_ff, d_model, bias=False)  # 降维回 d_model

    def forward(self, x):
        return self.W_down(F.silu(self.W_gate(x)) * self.W_up(x))  # ⊙ 是逐元素相乘

torch.manual_seed(0)
d_model = 64
x = torch.randn(2, 5, d_model)

# 老式 FFN：4× 宽，两个矩阵
vanilla = FeedForward(d_model, d_ff=4 * d_model, activation="gelu")
# SwiGLU：把 d_ff 收到 8/3 × d_model 来对齐参数量（取整到 8 的倍数更贴近真实实现）
d_ff_swiglu = round(8 / 3 * d_model / 8) * 8     # 8/3·64≈170.67，取整到 8 的倍数 -> 168
swiglu = SwiGLU(d_model, d_ff_swiglu)

y = swiglu(x)
n_vanilla = sum(p.numel() for p in vanilla.parameters())
n_swiglu = sum(p.numel() for p in swiglu.parameters())
print(f"SwiGLU 输出形状: {tuple(y.shape)}（形状守恒）")
print(f"老式 FFN   d_ff = {4*d_model:4d}（4×），参数量 = {n_vanilla}")
print(f"SwiGLU     d_ff = {d_ff_swiglu:4d}（≈8/3×），参数量 = {n_swiglu}")
print(f"→ SwiGLU 多一个矩阵，但靠把 d_ff 收到约 2/3，参数量和老式 4× FFN 基本持平。")

In [ ]:
# ============================================================
# Cell 6: 拼出一个完整的 Transformer block（Pre-LN + RMSNorm + SwiGLU）
# ============================================================
class MultiHeadSelfAttention(nn.Module):
    """紧凑版多头自注意力（第 7 章机制的精简实现），本章当成一个黑盒积木用。"""
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.H, self.d_k = num_heads, d_model // num_heads
        self.W_qkv = nn.Linear(d_model, 3 * d_model, bias=False)  # 一次投出 Q/K/V
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, L, D = x.shape
        qkv = self.W_qkv(x).reshape(B, L, 3, self.H, self.d_k).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]               # 各 [B, H, L, d_k]
        causal = torch.triu(torch.ones(L, L, dtype=torch.bool, device=x.device), 1)
        scores = (q @ k.transpose(-2, -1)) / self.d_k ** 0.5
        scores = scores.masked_fill(causal, float("-inf"))
        out = F.softmax(scores, dim=-1) @ v            # [B, H, L, d_k]
        out = out.transpose(1, 2).reshape(B, L, D)     # 合头
        return self.W_o(out)

class TransformerBlock(nn.Module):
    """现代 Decoder-only Transformer 层：Pre-LN，两个子层各带残差。"""
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.norm1 = MyRMSNorm(d_model)                # 注意力子层前的归一化
        self.attn = MultiHeadSelfAttention(d_model, num_heads)
        self.norm2 = MyRMSNorm(d_model)                # FFN 子层前的归一化
        self.ffn = SwiGLU(d_model, d_ff)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))               # 子层①：x + MHA(Norm(x))
        x = x + self.ffn(self.norm2(x))                # 子层②：x + FFN(Norm(x))
        return x

torch.manual_seed(0)
B, L, d_model, H = 2, 6, 64, 8
block = TransformerBlock(d_model, num_heads=H, d_ff=round(8/3*d_model/8)*8)
x = torch.randn(B, L, d_model)
out = block(x)
print("Transformer block 输入:", tuple(x.shape), " 输出:", tuple(out.shape), "（形状守恒）")
n_params = sum(p.numel() for p in block.parameters())
print(f"单层 block 参数量: {n_params}")
# 堆叠 N 层验证「积木」性质：输出形状始终不变
deep = nn.Sequential(*[TransformerBlock(d_model, H, round(8/3*d_model/8)*8) for _ in range(4)])
print("堆叠 4 层后输出:", tuple(deep(x).shape), "→ block 同形可无限堆叠")

In [ ]:
# ============================================================
# Cell 7: 读真实 Qwen3-8B config，看 FFN 宽度 / 激活 / 归一化
# ============================================================
# AutoConfig 只下载几 KB 的 config.json，不下载几个 GB 的权重，CPU 秒级完成。
from transformers import AutoConfig

cfg = AutoConfig.from_pretrained("Qwen/Qwen3-8B")
d_model = cfg.hidden_size
d_ff = cfg.intermediate_size          # FFN 中间宽度（SwiGLU 的 d_ff）
act = cfg.hidden_act                  # FFN 激活函数
eps = cfg.rms_norm_eps                # RMSNorm 的 ε

print(f"hidden_size (d_model)        : {d_model}")
print(f"intermediate_size (d_ff)     : {d_ff}")
print(f"d_ff / d_model               : {d_ff / d_model:.2f}  （SwiGLU 严格对齐参数是 8/3≈2.67）")
print(f"hidden_act (FFN 激活)        : {act}   ← silu，即 SwiGLU 的门控激活")
print(f"num_hidden_layers            : {cfg.num_hidden_layers}")
print(f"rms_norm_eps                 : {eps}   ← 用的是 RMSNorm，不是 LayerNorm")
print("-" * 56)
print("→ Qwen3-8B 的 FFN 是 SwiGLU（hidden_act=silu）、归一化是 RMSNorm，")
print(f"  d_ff 取到约 {d_ff / d_model:.1f}×d_model，略大于严格对齐的 8/3——多花点参数换表达能力。")